# Polars

"Polars is an open-source library for data manipulation, known for being one of the fastest data processing solutions on a single machine."

https://pola.rs/

Polars supports working with datasets that do not fit in memory by leveraging lazy evaluation, streaming execution, and efficient memory usage. 
- The core of Polars is written in Rust
- Polars stores data in columns

Main Polars object types:
- Series = a single column of data
- DataFrame = an in-memory, tabular dataset (eager execution); a collection of columns (Series)
- LazyFrame = a deferred execution plan, supports query optimization and streaming (lazy execution)
- Expr (Expression) = a computation or transformation to be applied to columns.

In [ ]:
# Install Polars (if necessary)

# !pip install polars

# You can also install Polars with optional features:
# https://docs.pola.rs/user-guide/installation/

# !pip install "polars[numpy, pandas]"

In [ ]:
import polars as pl
import datetime as dt

## Getting started

https://docs.pola.rs/user-guide/getting-started/#reading-writing

In [ ]:
df = pl.DataFrame(
    {
        "name": ["Alice Archer", "Ben Brown", "Chloe Cooper", "Daniel Donovan"],
        "birthdate": [
            dt.date(1997, 1, 10),
            dt.date(1985, 2, 15),
            dt.date(1983, 3, 22),
            dt.date(1981, 4, 30),
        ],
        "weight": [57.9, 72.5, 53.6, 83.1],  # (kg)
        "height": [1.56, 1.77, 1.65, 1.75],  # (m)
    }
)

In [ ]:
df.schema

In [ ]:
df

### Context: select

Expressions are computations or transformations that are performed on data columns.

Polars expressions need a context in which they are executed to produce a result. A context is the specific environment / situation in which an expression is evaluated.

In [ ]:
result = df.select(
    pl.col("name"),
    pl.col("birthdate").dt.year().alias("birth_year"),
    (pl.col("weight") / (pl.col("height") ** 2)).alias("bmi"),
)

result

### Context: with_columns


In [ ]:
result = df.with_columns(
    birth_year=pl.col("birthdate").dt.year(),
    bmi=pl.col("weight") / (pl.col("height") ** 2),
)

result

### Context: filter

In [ ]:
result = df.filter(pl.col("birthdate").dt.year() < 1990)

result

In [ ]:
result = df.filter(
    pl.col("birthdate").is_between(dt.date(1982, 12, 31), dt.date(1996, 1, 1)),
    pl.col("height") > 1.7,
)

result

### Context: group_by


In [ ]:
result = df.group_by(
    (pl.col("birthdate").dt.year() // 10 * 10).alias("decade"),
    maintain_order=True,
).len()

result

In [ ]:
# we can use agg to compute aggregations over the resulting groups

result = df.group_by(
    (pl.col("birthdate").dt.year() // 10 * 10).alias("decade"),
    maintain_order=True,
).agg(
    pl.len().alias("sample_size"),
    pl.col("weight").mean().round(2).alias("avg_weight"),
    pl.col("height").max().alias("tallest"),
)

result

## Titanic example

https://www.kaggle.com/datasets/markmedhat/titanic?resource=download

### Eager execution:

In [ ]:
df = pl.read_csv("data/titanic.csv")

In [ ]:
df.schema

In [ ]:
# first 5 rows
df.head()

In [ ]:
print("Summary statistics:\n")
df.describe()

In [ ]:
# count missing values in each column

missing_values = df.select([
    pl.col(col).is_null().sum().alias(col) for col in df.columns
])

print("Missing values:\n")
missing_values

In [ ]:
missing_values.glimpse()

In [ ]:
survival_counts = df.group_by("Survived").len()

print("Survival counts:\n")
survival_counts

In [ ]:
avg_age_by_class = df.group_by("Pclass").agg([
    pl.col("Age").mean().alias("Average_Age")
])

print("Average age by class:\n")
avg_age_by_class

In [ ]:
# Fill missing age with the median age

median_age = df.select(pl.col("Age").median()).item()

df = df.with_columns(
    pl.col("Age").fill_null(median_age)
)

In [ ]:
df.head()

In [ ]:
df.describe()

### Lazy execution

Polars supports two modes of operation: lazy and eager.

https://docs.pola.rs/user-guide/concepts/lazy-api/

Use `scan_csv` instead of `read_csv` to get lazy execution.

Benefits: Polars can use optimizations.

In the lazy API, the query is only evaluated once it is collected. Deferring the execution to the last minute can have significant performance advantages and is why the lazy API is preferred in most cases.

In [ ]:
df = pl.scan_csv("data/titanic.csv")

In [ ]:
type(df)

In [ ]:
df

In [ ]:
# Define the transformation pipeline

result = (
    df
    .with_columns([
        pl.col("Age").fill_null(pl.median("Age")).alias("Age"),
    ])
    .group_by("Pclass")
    .agg([
        pl.col("Age").mean().alias("Average_Age"),
        pl.col("Survived").mean().alias("Survival_Rate"),
        pl.len().alias("Count")
    ])
    .sort("Pclass")
)

In [ ]:
result

In [ ]:
print(result.explain())

In [ ]:
# Query is executed when it is collected

df_result = result.collect()

In [ ]:
df_result

## Resources

- [Polars: Getting Started](https://docs.pola.rs/user-guide/getting-started/)
- [Python Polars: A Lightning-Fast DataFrame Library](https://realpython.com/polars-python/) (Real Python)